# Embedding-model ablation on FinanceBench

Compares free, open-weight embedding models on the FinanceBench evidence-retrieval task, using the
same content-based relevance rule as the repo's harness (a passage is a hit if it contains every
numeric fact in the reference answer). GPU-bound models (gte-large, e5-large) need a GPU, so run this
on Colab.

**Setup:** Runtime → Change runtime type → **T4 GPU**.

**Version pin:** Colab's pre-installed `transformers` is too new for `gte-large`'s custom
`trust_remote_code` model (it triggers a `CUDA device-side assert`). The install cell pins compatible
versions — **after it runs, Runtime → Restart session once**, then run the remaining cells.

**If a model still asserts:** it corrupts the GPU context, so models after it also fail — restart and
re-run. The loop forces `max_seq_length = 512` (truncates long passages) and runs each model in its own
`try/except`, so one failure still leaves the earlier rows valid.

**Honest caveats:** these are *relative* numbers on a bespoke strict harness — **not** comparable to
MTEB/FinanceBench leaderboards. ~110 of 150 queries are content-scorable, so per-metric confidence
intervals are wide; read the *direction across k*, not single cells. e5 uses its required
`query:`/`passage:` prefixes; the BGE/GTE models are symmetric and need none.

Measured result (T4): bge-small `0.050 / 0.107 / 0.227`, bge-base `0.084 / 0.175 / 0.245`,
bge-large `0.060 / 0.109 / 0.218` (recall@1 / @5 / @10) — **bge-base wins; larger does not help.**

In [ ]:
# Pin versions compatible with gte-large's remote code, then RESTART SESSION before running on.
!pip -q install "transformers==4.44.2" "sentence-transformers==3.1.1" "datasets==3.0.1"
print("Installed. Now: Runtime -> Restart session, then run the cells below.")

In [ ]:
import re

import numpy as np
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer


# -- content-based relevance (matches frag.eval.relevance) --------------------
def _norm(t):
    return re.sub(r"\s+", "", t)


def extract_facts(ans):
    out = []
    for c in re.findall(r"\$?\d[\d,]*(?:\.\d+)?%?", ans or ""):
        if re.fullmatch(r"\d{4}", c):  # bare year, too common to be a key
            continue
        digits = re.sub(r"[^\d]", "", c)
        if len(digits) >= 3 or "." in c or "%" in c:
            out.append(_norm(c.lstrip("$")))
    return out


def relevant(text, facts):
    return bool(facts) and all(f in _norm(text) for f in facts)


# -- FinanceBench evidence passages + golden Q&A -----------------------------
ds = load_dataset("PatronusAI/financebench", split="train")
corpus, seen = [], set()
for row in ds:
    for ev in row.get("evidence") or []:
        t = (ev.get("evidence_text") or "").strip()
        if t and t not in seen:
            seen.add(t)
            corpus.append(t)
golden = [
    {"q": r["question"], "a": r["answer"]}
    for r in ds
    if r.get("question") and r.get("answer")
]
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"{len(corpus)} passages | {len(golden)} queries | {device}")

In [ ]:
# Each model: (label, name, kwargs, query_prefix, passage_prefix). e5 needs prefixes.
models = [
    ("bge-small", "BAAI/bge-small-en-v1.5", {}, "", ""),
    ("bge-base", "BAAI/bge-base-en-v1.5", {}, "", ""),
    ("bge-large", "BAAI/bge-large-en-v1.5", {}, "", ""),
    ("gte-large", "Alibaba-NLP/gte-large-en-v1.5", {"trust_remote_code": True}, "", ""),
    ("e5-large", "intfloat/e5-large-v2", {}, "query: ", "passage: "),
]

print(f"\n{'model':12}{'recall@1':>10}{'recall@5':>10}{'recall@10':>11}")
for label, name, kw, qpx, ppx in models:
    try:
        m = SentenceTransformer(name, device=device, **kw)
        m.max_seq_length = 512  # truncate long passages -> avoids CUDA index asserts
        de = m.encode([ppx + t for t in corpus], normalize_embeddings=True, batch_size=32,
                      convert_to_numpy=True, show_progress_bar=False)
        qe = m.encode([qpx + g["q"] for g in golden], normalize_embeddings=True, batch_size=32,
                      convert_to_numpy=True, show_progress_bar=False)
        sims = qe @ de.T
        rec = {1: 0.0, 5: 0.0, 10: 0.0}
        n = 0
        for i, g in enumerate(golden):
            facts = extract_facts(g["a"])
            if not facts:
                continue
            n += 1
            top10 = np.argsort(-sims[i])[:10]
            goldpos = [p for p, j in enumerate(top10) if relevant(corpus[int(j)], facts)]
            for k in rec:
                if goldpos:
                    rec[k] += sum(1 for p in goldpos if p < k) / len(goldpos)
        print(f"{label:12}{rec[1] / n:10.3f}{rec[5] / n:10.3f}{rec[10] / n:11.3f}")
        del m
        torch.cuda.empty_cache()
    except Exception as e:
        print(f"{label:12}  FAILED: {type(e).__name__}: {str(e)[:60]}")